In [1]:
import pickle
import pandas as pd
from pathlib import Path

In [8]:
categorical = ['PULocationID', 'DOLocationID']


def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

def load_artifacts(file):
    with open(file, 'rb') as f_in:
        dv, model = pickle.load(f_in)
    return dv, model

def gen_result(df, dv, model):
    dicts = df[categorical].to_dict(orient='records')
    X_val = dv.transform(dicts)
    y_pred = model.predict(X_val)
    df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')
    df['predicted_duration']=y_pred 
    df_result=df.loc[:,['ride_id', 'predicted_duration']]
    return df_result

def export_result(df_result, year, month):
    # Build and create the directory
    output_folder = Path("./output")
    output_folder.mkdir(parents=True, exist_ok=True)  # parents=True is harmless here

    # Construct the full path in an object-oriented way
    output_file = output_folder / f"homework_preds_{year}_{month:02d}.parquet"

    df_result.to_parquet(
        output_file,
        engine='pyarrow',
        compression=None,
        index=False
    )
    
def print_mean_pred(df_result):
    print(df_result['predicted_duration'].mean())

In [9]:
year=2023
month=3
dv, model=load_artifacts('model.bin')
df = read_data(f'yellow_tripdata_{year}-{month:02d}.parquet')
df_result=gen_result(df, dv, model)
export_result(df_result, year, month)
print_mean_pred(df_result)

/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


14.203865642696083
